

# Studio 1 · Startup Runway Casino - Starter
### OPIM 5641 - Business Decision Modeling · In-person lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/Studios/Studio1_StartupCasino/Startup_Casino_starter.ipynb)

*Save your own copy first (File → Save a copy in GitHub or Drive), then make the assumptions YOURS.*

## The game
You just founded a startup. You have some cash in the bank, revenue that *might* show up, and expenses that definitely will. **Question: what's the probability you survive the next 12 months?**

Every founder in the room (that's you) will:
1. Write down their **assumptions** - starting cash, monthly revenue (a distribution!), monthly burn (a distribution!)
2. Get ONE month right, then ONE year
3. Simulate **10,000 parallel universes** of their startup
4. Report **P(survive 12 months)** - the share of universes where cash never goes below zero

Then we put every founder's number on the board: the **leaderboard**. Highest survival probability with a *defensible* story wins. (A startup with \$10M in the bank and \$1 of burn survives everything and impresses no one - your assumptions have to pass the reasonable-person test.)

🔷 **The nugget:** this is the retirement simulation wearing a hoodie - same recipe: describe uncertainty → one trial right → loop 10,000 → read the distribution.

In [ ]:
# import our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Step 1 - YOUR assumptions
The defaults below run, but they're MY startup, not yours. Change them - pick distributions you can defend out loud. (Triangular for expert guesses, normal for sums-of-many-things, uniform for genuine no-idea - remember?)

In [ ]:
# TODO: make these YOURS - and be ready to defend every number
starting_cash = 90000    # $ in the bank on day 1 - my seed round was modest
months = 12              # the runway we care about

def monthly_revenue():
    # my startup: revenue is a hopeful triangle - usually ~8K, could be 0, could pop to 30K
    return np.random.triangular(left=0, mode=8000, right=30000)

def monthly_burn():
    # my startup: payroll+rent+cloud, normally distributed around 18K, sd 3K
    return np.random.normal(loc=18000, scale=3000)

## Step 2 - get ONE month right, then ONE year
One month: cash = cash + revenue - burn. One year: 12 months in a for loop, carrying cash forward (the handoff!). If cash dips below zero at ANY point, the startup is dead - it doesn't get to un-die because December was great.

In [ ]:
# one year of one universe - run this a few times and watch your fate change
cash = starting_cash
alive = True
for m in np.arange(0, months, 1):
    cash = cash + monthly_revenue() - monthly_burn() # the handoff
    if cash < 0:
        alive = False
        break # dead is dead - stop the year
print('ending cash:', round(cash), '| survived?', alive)

## Step 3 - 10,000 parallel universes
Wrap the year in an outer loop. Store TWO things per universe: did it survive, and the ending cash (for survivors).

In [ ]:
# the full casino - 10,000 startups
survived = []      # True/False per universe
ending_cash = []   # ending cash per universe (dead startups record their moment of death)
for b in np.arange(0, 10000, 1):
    cash = starting_cash
    alive = True
    for m in np.arange(0, months, 1):
        cash = cash + monthly_revenue() - monthly_burn()
        if cash < 0:
            alive = False
            break
    survived.append(alive)
    ending_cash.append(cash)

survived = np.array(survived)
ending_cash = np.array(ending_cash)
print('universes:', len(survived))

## Step 4 - the leaderboard number (and the story behind it)
P(survive) is the headline - but a founder who can also say "and in the median universe I end the year with \$X" is the one who gets funded.

In [ ]:
# THE leaderboard number
p_survive = survived.mean()
print(f'P(survive 12 months) = {p_survive:.1%}')

# the fuller story - percentiles of ending cash
print(pd.Series(ending_cash).quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(0))

In [ ]:
# the picture - where do the 10,000 universes end up?
plt.hist(ending_cash, bins=100, color='darkorange', edgecolor='black')
plt.axvline(0, color='red', linewidth=2, label='broke')
plt.title(f'Startup Runway Casino - P(survive) = {p_survive:.1%}')
plt.xlabel('Cash at month 12 (or at death)')
plt.ylabel('Universes')
plt.legend()
plt.show()

## Step 5 - play the casino
- Write your **P(survive)** on the board with your startup's one-line pitch.
- **On your own experiments:** What single change to your assumptions buys the most survival - more starting cash, higher revenue mode, or lower burn sd? Change ONE thing at a time and re-run. That habit has a name now: sensitivity analysis. It's where this course is headed.
- Commit your notebook to your repo before you leave.

**Caution:** no seeds in the loop! If your P(survive) is exactly the same on every run, you've frozen the randomness - go find the seed and take it out.